<h1 style="text-align:center;">Lab 5 — FashionMNIST CNN · Обучение</h1>

Этот ноутбук рассчитан на запуск в **Google Colab с GPU** (Runtime → Change runtime type → GPU).

**Что делает:**
1. Скачивает FashionMNIST.
2. Обучает свёрточную сеть с аугментациями.
3. Сохраняет лучшие веса в `best_convnet.pt` и историю обучения в `history.json`.

После обучения оба файла нужно скачать локально и положить рядом с `Lab5_convnet_infer.ipynb`.

In [ ]:
# !pip install torchvision

import json
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Используемое устройство:', device)


## Данные

Классы FashionMNIST: T-shirt/top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Ankle boot.

In [ ]:
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
           'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')

# Нормировка по среднему и std FashionMNIST
MEAN, STD = (0.2860,), (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

trainset = torchvision.datasets.FashionMNIST(root='./data', train=True,
                                             download=True, transform=train_transform)
testset = torchvision.datasets.FashionMNIST(root='./data', train=False,
                                            download=True, transform=test_transform)

BATCH_SIZE = 128
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True,
                         num_workers=2, pin_memory=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

print(f'Train: {len(trainset)} | Test: {len(testset)} | Классов: {len(classes)}')


In [ ]:
# Быстрый взгляд на по одной картинке каждого класса
raw_view = torchvision.datasets.FashionMNIST(root='./data', train=True,
                                             download=True, transform=transforms.ToTensor())
labels_np = np.array(raw_view.targets)

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for cls_id, ax in enumerate(axes.ravel()):
    idx = np.where(labels_np == cls_id)[0][0]
    img, _ = raw_view[idx]
    ax.imshow(img.squeeze().numpy(), cmap='gray')
    ax.set_title(classes[cls_id])
    ax.axis('off')
plt.tight_layout()
plt.show()


## Архитектура

VGG-style CNN для 28×28: три блока `[Conv-BN-ReLU] × 2 → MaxPool → Dropout`, дальше FC-голова.

In [ ]:
class ConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 1 -> 32, 28x28
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 2: 32 -> 64, 14x14
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 3: 64 -> 128, 7x7
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        # 28 -> 14 -> 7 -> 3, каналов 128, итого 128*3*3 = 1152
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


net = ConvNet().to(device)
n_params = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f'Число обучаемых параметров: {n_params:,}')


## Обучение

In [ ]:
NUM_EPOCHS = 20
LR = 1e-3

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss_sum += loss_fn(logits, y).item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += X.size(0)
    return loss_sum / total, correct / total


history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    net.train()
    running_loss, running_correct, running_total = 0.0, 0, 0
    for X, y in trainloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = net(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += X.size(0)

    scheduler.step()

    train_loss = running_loss / running_total
    train_acc = running_correct / running_total
    test_loss, test_acc = evaluate(net, testloader)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(net.state_dict(), 'best_convnet.pt')

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
          f'train loss {train_loss:.4f} acc {train_acc:.4f} | '
          f'test loss {test_loss:.4f} acc {test_acc:.4f} | '
          f'lr {scheduler.get_last_lr()[0]:.5f}')

print(f'\nЛучшая accuracy на тесте: {best_acc:.4f}')


## Сохраняем артефакты

После этой ячейки нужно скачать оба файла из Colab (вкладка Files слева → правый клик → Download).

In [ ]:
# Сохраняем историю обучения для отрисовки в инференс-ноутбуке
with open('history.json', 'w') as f:
    json.dump(history, f, indent=2)

print('Сохранено:')
print('  best_convnet.pt — веса лучшей модели')
print('  history.json    — история обучения (для графиков)')

# Если в Colab — можно сразу скачать:
# from google.colab import files
# files.download('best_convnet.pt')
# files.download('history.json')
